# Credit Card Fraud Detection

## 1. Data Loading and Exploration

In [ ]:
import numpy as np
import pandas as pd

fraud = pd.read_csv("../data/creditcard.csv")
fraud.shape


In [ ]:
fraud.describe()

In [ ]:
fraud.info()

Creating a copy of the orginal dataset

In [ ]:
fraud_copy = fraud.copy()


## 2. Data Preprocessing

### Train-Test Split (Stratified)
Using stratified split to preserve class distribution (0.17% fraud rate)

In [ ]:
from sklearn.model_selection import train_test_split

train_fraud, test_fraud = train_test_split(fraud_copy, test_size=0.2, random_state=42, stratify = fraud_copy["Class"])



In [ ]:
train_fraud

In [ ]:
test_fraud

### Extracting Input and Target Features
Separating features (V1-V28, Time, Amount) from target variable (Class) for train and test sets

In [ ]:
def extract(df, input_cols, target_col):
    input = df[input_cols].copy()
    target = df[target_col].copy()
    return input, target

input_cols = fraud_copy.columns[0: -1]
target_col = "Class"

train_input, train_target = extract(train_fraud, input_cols, target_col)
test_input, test_target = extract(test_fraud, input_cols, target_col)

train_input

In [ ]:
train_input

test_input

In [ ]:
test_input

train_target

In [ ]:
train_target

test_target

In [ ]:
test_target

In [ ]:
train_input.isna().sum()


### 2.2 Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(train_input)

train_input = scaler.transform(train_input)
test_input = scaler.transform(test_input)

## 3. Model 1: Logistic Regression (Baseline)

### Training

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000, class_weight = 'balanced', random_state=42)
model_lr.fit(train_input, train_target)



### Evaluation

In [ ]:
train_preds = model_lr.predict(train_input)

In [ ]:
test_preds = model_lr.predict(test_input)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(train_target, train_preds)

In [ ]:
print(model_lr.classes_)
train_proba = model_lr.predict_proba(train_input)
train_proba

Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(test_target, test_preds, normalize='true')

Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_target, test_preds))

### Baseline Comparison

In [ ]:
def dumb_model(set):
    return np.random.choice([0, 1], len(set))


print("accuracy_score when the prediction is randomly filled with 0s and 1s:   ", accuracy_score(test_target, dumb_model(test_input)))

def all_no(set):
    return np.full(len(set), 0)

print("accuracy_score when all the predcitions are 0s i.e., no fraud happend:  ", accuracy_score(test_target, all_no(test_input)))

def all_yes(set):
    return np.full(len(set), 1)

print("accuracy_score when all the predcitions are 1s i.e., when fraud happend:", accuracy_score(test_target, all_yes(test_input)))

print("\n=== Baseline Comparison ===")
print("Random guess accuracy(0 or 1): ", accuracy_score(train_target, dumb_model(train_input)))
print("Always predicts 'No Fraud'(0): ", accuracy_score(train_target, all_no(train_input)))
print("Always predicts 'Fraud'(1)     ", accuracy_score(train_target, all_yes(train_input)))
print("My LR model:                   ", accuracy_score(test_target, test_preds))
print("\nAccuracy is misleading for imbalanced data.")
print("LR fraud recall: 92%")
print("Baseline recall: 0%")


### PR-AUC

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

auc_lr = model_lr.predict_proba(test_input)[:, 1]

precision, recall, thresholds = precision_recall_curve(test_target, auc_lr)
pr_auc = auc(recall, precision)

print("PR-AUC_LR:", pr_auc)

## 4. Model 2: Random Forest
As established earlier, accuracy is misleading for this imbalanced dataset. Evaluating with precision-recall metrics.

### Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators = 100,  max_depth = 5, n_jobs = -1, class_weight = 'balanced', random_state = 42)
model_rf.fit(train_input, train_target)

### Evaluation

In [ ]:
train_pred_rf = model_rf.predict(train_input)

In [ ]:
test_pred_rf = model_rf.predict(test_input)

Confusion Matrix

In [ ]:
confusion_matrix(test_target, test_pred_rf, normalize='true')

Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_target, test_pred_rf))

### Feature Importance

In [ ]:
importance_rf = pd.DataFrame({
    'feature': input_cols,
    'importance': model_rf.feature_importances_
}).sort_values('importance', ascending=False)
importance_rf.head(10)

### PR-AUC

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

auc_rf = model_rf.predict_proba(test_input)[:, 1]

precision, recall, thresholds = precision_recall_curve(test_target, auc_rf)
pr_auc = auc(recall, precision)

print("PR-AUC_RF:", pr_auc)

## 5. Model 3: XGBoost (Best Model)
As established earlier, accuracy is misleading for this imbalanced dataset. Evaluating with precision-recall metrics.

### Training

In [ ]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(n_estimators = 200,  max_depth = 5, n_jobs = -1, learning_rate = 0.5, scale_pos_weight = 25, eval_metric = 'logloss', random_state = 42)
model_xgb.fit(train_input, train_target)

### Evaluation

In [ ]:
train_pred_xgb = model_xgb.predict(train_input)

In [ ]:
test_pred_xgb = model_xgb.predict(test_input)

Confusion Matrix

In [ ]:
confusion_matrix(test_target, test_pred_xgb, normalize='true')

Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_target, test_pred_xgb))

In [ ]:
importance_xgb = pd.DataFrame({
    'feature': input_cols,
    'importance': model_xgb.feature_importances_
}).sort_values('importance', ascending=False)
importance_xgb.head(10)

### PR-AUC

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

auc_xgb = model_xgb.predict_proba(test_input)[:, 1]

precision, recall, thresholds = precision_recall_curve(test_target, auc_xgb)
pr_auc = auc(recall, precision)

print("PR-AUC_XGB:", pr_auc)


## 6. Final Model Comparison

| Model | Precision (Fraud) | Recall (Fraud) | F1-Score | Key Insight |
|-------|-------------------|----------------|----------|-------------|
| Logistic Regression | 0.06 | 0.92 | 0.11 | Catches 92% fraud but 94% false alarm rate - unusable |
| Random Forest | 0.36 | 0.89 | 0.51 | Better than LR but still 64% false alarms |
| XGBoost | 0.91 | 0.84 | 0.87 | Best balance - only 9% false alarms, catches 84% fraud |

### Conclusion
XGBoost achieved optimal precision-recall balance for production deployment. While it catches slightly fewer frauds than LR (84% vs 92%), it dramatically reduces false positives (91% precision vs 6%), making it the clear choice for real-world application where false alarms damage customer trust. XGBoost also achieved the highest PR-AUC (0.878), confirming superior performance across probability thresholds under extreme class imbalance.

## Saving the model

In [ ]:
import joblib

joblib.dump(model_xgb, "Best_model.pkl")
joblib.dump(scaler, "Scaler.pkl")

## SQL Analysis
Loaded transaction data into SQLite to perform business-level analysis using SQL queries.
Key findings:
- Overall fraud rate: 0.17% (492/284,807 transactions)
- High-value transactions (>$200) show 2x higher fraud rate (0.29%)
- Fraudulent transactions have higher average amount ($122 vs $88)
- Peak fraud hours: 2AM-3AM with 1.33% fraud rate

In [ ]:
import sqlite3

conn = sqlite3.connect("../data/fraud.db")
df = pd.read_csv("../data/creditcard.csv")
df.to_sql("transactions", conn, if_exists="replace", index=False)

In [ ]:
queries = {
    "1. Fraud Rate": """
        SELECT COUNT(*) AS total, SUM(Class) AS frauds,
               ROUND(100.0 * SUM(Class) / COUNT(*), 4) AS fraud_rate
        FROM transactions
    """,
    "2. Fraud by Amount Bucket": """
        SELECT CASE WHEN Amount < 50 THEN 'Low (<$50)'
                    WHEN Amount < 200 THEN 'Medium ($50-200)'
                    ELSE 'High (>$200)' END AS amount_bucket,
               COUNT(*) AS total, SUM(Class) AS frauds,
               ROUND(100.0 * SUM(Class) / COUNT(*), 2) AS fraud_rate
        FROM transactions GROUP BY amount_bucket
    """,
    "3. Avg Amount (Fraud vs Normal)": """
        SELECT Class, ROUND(AVG(Amount), 2) AS avg_amount,
               ROUND(MAX(Amount), 2) AS max_amount
        FROM transactions GROUP BY Class
    """,
    "4. Top 10 Suspicious Transactions": """
        SELECT Time, Amount, V1, V14, V17
        FROM transactions WHERE Class = 1
        ORDER BY Amount DESC LIMIT 10
    """,
    "5. Hourly Fraud Pattern": """
        SELECT CAST(Time/3600 AS INT) AS hour,
               COUNT(*) AS total, SUM(Class) AS frauds,
               ROUND(100.0 * SUM(Class) / COUNT(*), 2) AS fraud_rate
        FROM transactions GROUP BY hour ORDER BY hour
    """
}

for title, query in queries.items():
    print(f"{title}")
    display(pd.read_sql(query, conn))

conn.close()